In [1]:
import pandas as pd, numpy as np

df = pd.read_csv('../data/AmesHousing.csv')
df.columns = df.columns.str.replace(r'[ /]', '', regex=True)  # "Gr Liv Area" -> "GrLivArea"
print(df.shape)
df.head()

(2930, 82)


,Order,PID,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [2]:
cols_num = ['GrLivArea', 'BedroomAbvGr', 'FullBath', 'HalfBath',
            'YearBuilt', 'YearRemodAdd', 'OverallQual', 'GarageCars']
cols_cat = ['Neighborhood']

X = df[cols_num + cols_cat]
y = np.log(df['SalePrice'])   # el diseño predice el logaritmo del precio

print(X.shape)
print(X.isna().sum())

(2930, 9)
GrLivArea       0
BedroomAbvGr    0
FullBath        0
HalfBath        0
YearBuilt       0
YearRemodAdd    0
OverallQual     0
GarageCars      1
Neighborhood    0
dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Entrenamiento:", len(X_train), "casas")
print("Prueba:", len(X_test), "casas")

Entrenamiento: 2344 casas
Prueba: 586 casas


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor

prep = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), cols_num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cols_cat),
])

modelo = Pipeline([
    ('prep', prep),
    ('gb', GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                     max_depth=3, random_state=42)),
])

modelo.fit(X_train, y_train)
print("Modelo entrenado")

Modelo entrenado


In [5]:
from sklearn.metrics import r2_score, mean_absolute_percentage_error

pred = np.exp(modelo.predict(X_test))   # volver de logaritmo a dólares
real = np.exp(y_test)

mape = mean_absolute_percentage_error(real, pred) * 100
r2 = r2_score(real, pred)

print(f"MAPE: {mape:.2f} %")
print(f"R2:   {r2:.3f}")

MAPE: 10.60 %
R2:   0.878


In [6]:
from sklearn.model_selection import cross_val_predict

pred_cv = np.exp(cross_val_predict(modelo, X_train, y_train, cv=5))
mape_cv = mean_absolute_percentage_error(np.exp(y_train), pred_cv) * 100
print(f"MAPE validación cruzada (solo entrenamiento): {mape_cv:.2f} %")

MAPE validación cruzada (solo entrenamiento): 11.09 %


In [7]:
def evaluar_cv(cols_num, cols_cat):
    cols = cols_num + cols_cat
    Xtr = df.loc[X_train.index, cols]
    prep = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), cols_num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cols_cat),
    ])
    m = Pipeline([
        ('prep', prep),
        ('gb', GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                         max_depth=3, random_state=42)),
    ])
    pred = np.exp(cross_val_predict(m, Xtr, y_train, cv=5))
    return mean_absolute_percentage_error(np.exp(y_train), pred) * 100

print("Base (9 variables):  ", round(evaluar_cv(cols_num, cols_cat), 2), "%")
print("+ OverallCond:       ", round(evaluar_cv(cols_num + ['OverallCond'], cols_cat), 2), "%")

Base (9 variables):   11.09 %
+ OverallCond:        10.81 %


In [8]:
num_base = cols_num + ['OverallCond']

print("Base + OverallCond:", round(evaluar_cv(num_base, cols_cat), 2), "%")

for v in ['TotalBsmtSF', 'LotArea', 'Fireplaces']:
    print(f"+ {v}:", round(evaluar_cv(num_base + [v], cols_cat), 2), "%")

for v in ['ExterQual', 'KitchenQual']:
    print(f"+ {v}:", round(evaluar_cv(num_base, cols_cat + [v]), 2), "%")

Base + OverallCond: 10.81 %
+ TotalBsmtSF: 10.07 %
+ LotArea: 10.35 %
+ Fireplaces: 10.62 %
+ ExterQual: 10.78 %
+ KitchenQual: 10.7 %


In [9]:
num_A = num_base + ['TotalBsmtSF', 'LotArea', 'Fireplaces']

print("Combinación A:", round(evaluar_cv(num_A, cols_cat), 2), "%")
print("Combinación B (A + KitchenQual):",
      round(evaluar_cv(num_A, cols_cat + ['KitchenQual']), 2), "%")

Combinación A: 9.63 %
Combinación B (A + KitchenQual): 9.57 %


In [10]:
import joblib, json

cols_num_final = num_A
cols_cat_final = cols_cat

X_train_f = df.loc[X_train.index, cols_num_final + cols_cat_final]
X_test_f  = df.loc[X_test.index,  cols_num_final + cols_cat_final]

prep_f = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), cols_num_final),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cols_cat_final),
])
modelo_final = Pipeline([
    ('prep', prep_f),
    ('gb', GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                     max_depth=3, random_state=42)),
])
modelo_final.fit(X_train_f, y_train)

pred = np.exp(modelo_final.predict(X_test_f))
real = np.exp(y_test)
mape = mean_absolute_percentage_error(real, pred) * 100
r2 = r2_score(real, pred)
print(f"MAPE en prueba: {mape:.2f} %")
print(f"R2 en prueba:   {r2:.3f}")

# guardar el modelo y la lista de variables
joblib.dump(modelo_final, '../models/valuia.joblib')
with open('../models/columnas.json', 'w') as f:
    json.dump({'num': cols_num_final, 'cat': cols_cat_final}, f)
print("Modelo y columnas guardados en la carpeta models")

MAPE en prueba: 8.20 %
R2 en prueba:   0.919
Modelo y columnas guardados en la carpeta models
